In [4]:
import os
import sys

import geopandas as gpd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

from src.variables import build_classe_taille, build_part, build_profil, build_tcam, build_trajectoire

communes = gpd.read_parquet("data/clean/base_finale/base_finale_communes.parquet")
nb_annees = config["annee_fin"] - config["annee_debut"]

In [5]:
communes["tcam_pop_2017_2023"] = build_tcam(communes["pop_2017"], communes["pop_2023"], nb_annees)
communes["trajectoire"] = build_trajectoire(communes["tcam_pop_2017_2023"], config["seuil_stable_pct"])

print(communes["tcam_pop_2017_2023"].describe())
print(communes["trajectoire"].value_counts(dropna=False))

count    34740.000000
mean         0.077497
std          1.466141
min        -12.429141
25%         -0.717883
50%          0.036597
75%          0.828529
max         18.364677
Name: tcam_pop_2017_2023, dtype: float64
trajectoire
croissance    15447
déclin        14405
stable         4888
NaN               6
Name: count, dtype: int64


In [6]:
# croissance des voyageurs des gares de la commune (NaN sans gare)
communes["tcam_voyageurs_2017_2023"] = build_tcam(communes["voyageurs_2017"], communes["voyageurs_2023"], nb_annees)

# voyageurs des gares de la commune par habitant (NaN sans gare)
communes["voyageurs_hab_2023"] = communes["voyageurs_2023"] / communes["pop_2023"].where(communes["pop_2023"] > 0)

# part des actifs allant travailler en voiture, NaN sous 1 000 actifs
for annee in [config["annee_debut"], config["annee_fin"]]:
    communes[f"part_voiture_{annee}"] = build_part(communes[f"actifs_voiture_{annee}"], communes[f"actifs_{annee}"], config["actifs_min"])

# part de la voiture dans toutes les communes, sans seuil d'actifs (pour les boîtes par taille de commune)
communes["part_voiture_toutes_2023"] = build_part(communes["actifs_voiture_2023"], communes["actifs_2023"], 1)

# voitures des particuliers pour 1 000 habitants
communes["voitures_1000_hab_2023"] = communes["voitures_2023"] / communes["pop_2023"].where(communes["pop_2023"] > 0) * 1000

In [7]:
communes["densite"] = communes["pop_2023"] / communes["superficie_km2"]
communes["classe_taille"] = build_classe_taille(communes["pop_2023"], config["bornes_taille"], config["noms_taille"])
communes["profil"], communes["desservie"] = build_profil(communes["tcam_pop_2017_2023"], communes["dist_gare_km"], config["seuil_desservie_km"])

print(communes["classe_taille"].value_counts().sort_index())
print(communes["profil"].value_counts(dropna=False))

classe_taille
moins de 500       18252
500 à 2 000        11097
2 000 à 5 000       3240
5 000 à 10 000      1169
10 000 à 50 000      865
50 000 et plus       123
Name: count, dtype: int64
profil
croissance – desservie              10777
déclin ou stable – desservie         9240
déclin ou stable – non desservie     7752
croissance – non desservie           6611
NaN                                   366
Name: count, dtype: int64


In [8]:
communes.to_parquet("data/clean/base_finale/base_finale_communes_variables.parquet")
print(communes.shape)

(34746, 48)
